In [1]:
!pip install -q langchain==0.1.20 langchain-google-genai langchain-community tavily-python chromadb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.1/679.1 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 13.3 MB/s eta

In [2]:
import os
from getpass import getpass

# Configurar API keys de forma segura
print("Configuración de API Keys")
os.environ["GOOGLE_API_KEY"] = getpass("Introduce tu Google API key: ")
os.environ["TAVILY_API_KEY"] = getpass("Introduce tu Tavily API key: ")
print("API Keys configuradas correctamente\n")


Configuración de API Keys
Introduce tu Google API key: ··········
Introduce tu Tavily API key: ··········
API Keys configuradas correctamente



# EJERCICIO 1: MEMORIA CONVERSACIONAL - BUFFER
Por defecto, los modelos de lenguaje (LLMs) no tienen memoria:
cada interacción es independiente y el modelo no recuerda lo dicho previamente.

La principal ventaja de la Buffer Memory es que permite mantener una conversación coherente y contextual, ya que conserva todo el historial y el modelo puede “recordar” exactamente lo que se ha dicho. Sin embargo, su desventaja es que al almacenar toda la conversación consume un gran número de tokens y aumenta el coste computacional, además de poder incluir información irrelevante o redundante que con el tiempo degrade la calidad de las respuestas. continuidad en sus respuestas.

In [3]:

# Imports necesarios para este ejercicio
from langchain_google_genai import ChatGoogleGenerativeAI #Librería de Langchain de Google, maneja la conversación
from langchain.memory import ConversationBufferMemory # Para dar memoria al chatbot
from langchain.chains import ConversationChain # Gestiona el prcoes de una conversación entre LLM y Memoria

# Inicializar el modelo de lenguaje Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-exp",  # Modelo Gemini 2.0 Flash (rápido y eficiente)
    temperature=0  # Respuestas deterministas (menos creativas, más consistentes)
)
print("LLM Gemini inicializado correctamente\n")

# Crear memoria buffer, y almaceno todos los mensajes temporales
buffer_memory = ConversationBufferMemory()

# Crear cadena conversacional con memoria
conversation_buffer = ConversationChain(
    llm=llm,  # El modelo que usará
    memory=buffer_memory,  # La memoria que recordará la conversación
    verbose=True  # Mostrar el proceso interno (útil para aprender)
)

# Conversación de ejemplo
print("\nUsuario: Hola, me llamo Juan")

print(f"Asistente: {respuesta1}\n")

print("Usuario: ¿Cuál es mi nombre?")
respuesta2 = conversation_buffer.predict(input="¿Cuál es mi nombre?")
print(f"Asistente: {respuesta2}\n")

# Inspeccionar qué hay en la memoria
print("Contenido de la memoria buffer:")
print(buffer_memory.load_memory_variables({}))
print("\n")



LLM Gemini inicializado correctamente


Usuario: Hola, me llamo Juan


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hola, me llamo Juan
AI:

> Finished chain.
Asistente: Hola Juan! Mucho gusto. My name is, well, I don't really *have* a name in the human sense. You can call me AI, or just refer to me as the language model. I'm glad to be chatting with you. I'm running on a large language model, trained by Google. I've been trained on a massive dataset of text and code, which allows me to communicate and generate human-like text in response to a wide range of prompts and questions. So, Juan, what can I do for you today?


Usuario: ¿Cuál es mi nombre?


> Entering new ConversationChain chain...
Prompt after fo

# EJERCICIO 2: MEMORIA CONVERSACIONAL - RESUMEN
La memoria tipo Summary resume de forma automática las conversaciones largas para mantener el contexto sin necesidad de almacenar todo el historial. Su principal ventaja es que reduce el uso de tokens, ya que conserva solo una versión condensada de lo anterior. Sin embargo, este enfoque tiene como desventaja que puede perder matices o detalles específicos presentes en los mensajes originales, lo que a veces afecta la precisión o coherencia en respuestas posteriores.

In [5]:

# Import adicional para memoria con resumen
from langchain.memory import ConversationSummaryMemory

# Crear memoria con resumen automático
# Le digo que modelo o LLM usar para hacer resúmenes
summary_memory = ConversationSummaryMemory(llm=llm)

# Contenedor de Langchain para manejar diálogos con un LLM
conversation_summary = ConversationChain(
    llm=llm,
    memory=summary_memory, # Usa el modo resumen, extrae inserta y resume, luego actualiza
    verbose=True # Flag que controla lo que vemos internamente
)

# Muestro por pantalla para que se vea la pregunta
print("\nUsuario: Hola, trabajo como profesor de IA")
# Usamos nuestra pregunta estilo resumen
respuesta1 = conversation_summary.predict(input="Hola, trabajo como profesor de IA")
print(f"Asistente: {respuesta1}\n")

print("Usuario: Voy a dar una clase sobre agentes mañana")
respuesta2 = conversation_summary.predict(input="Voy a dar una clase sobre agentes mañana")
print(f"Asistente: {respuesta2}\n")

print("Usuario: Necesito preparar ejemplos de RAG")
respuesta3 = conversation_summary.predict(input="Necesito preparar ejemplos de RAG")
print(f"Asistente: {respuesta3}\n")

print("Usuario: ¿Qué sabes de mí?")
respuesta4 = conversation_summary.predict(input="¿Qué sabes de mí?")
print(f"Asistente: {respuesta4}\n")

# Mostrar el resumen generado
print("Resumen de la conversación:")
print(summary_memory.load_memory_variables({}))
print("\n")



Usuario: Hola, trabajo como profesor de IA


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hola, trabajo como profesor de IA
AI:

> Finished chain.
Asistente: ¡Hola! ¡Qué interesante! Trabajar como profesor de IA debe ser muy estimulante. ¿En qué área de la IA te especializas? ¿Enseñas aprendizaje automático, procesamiento del lenguaje natural, visión artificial, o algo más?

Me imagino que tus estudiantes deben estar muy motivados, dado el auge de la IA en los últimos años. ¿Qué tipo de proyectos suelen realizar tus alumnos? ¿Trabajan con frameworks como TensorFlow o PyTorch?

Yo, como modelo de lenguaje grande, estoy entrenado en una gran cantidad de texto y código. Puedo generar texto, traducir idiomas, 

# EJERCICIO 3: RAG - MEMORIA DOCUMENTAL
El enfoque RAG (Retrieval-Augmented Generation) permite que un modelo de lenguaje acceda a información externa y actualizada almacenada en documentos propios. El proceso consiste en dividir el texto en fragmentos, generar sus embeddings y guardarlos en una base vectorial. Ante una consulta, el sistema recupera los fragmentos más relevantes y los ofrece al modelo como contexto, para que genere una respuesta precisa y fundamentada sin necesidad de reentrenamiento.

In [11]:
!pip install faiss-cpu
# Imports necesarios para RAG
from langchain_google_genai import GoogleGenerativeAIEmbeddings # Embeddings para transfomar embeddings
from langchain.chains import RetrievalQA # Constructor del RAG
from langchain_community.document_loaders import TextLoader # Carga los textos
from langchain.text_splitter import CharacterTextSplitter # Para dividir caracteres en chunks
from langchain_community.vectorstores import FAISS  # Almacén de embbedings

# PASO 1: Crear documentos de ejemplo
print("\nCreando documentos de ejemplo...")

# Documento 1: Sobre LangChain
with open("doc1.txt", "w", encoding="utf-8") as f:
    f.write("""LangChain es un framework para desarrollar aplicaciones con LLMs.
Permite crear cadenas, agentes y sistemas RAG. Es muy útil para construir
aplicaciones inteligentes que necesitan acceso a datos externos.""")

# Documento 2: Sobre Agentes
with open("doc2.txt", "w", encoding="utf-8") as f:
    f.write("""Los agentes en LangChain pueden usar herramientas como búsqueda web,
calculadoras y bases de datos. Un agente decide qué herramienta usar según
la tarea que debe resolver.""")

# Documento 3: Sobre RAG
with open("doc3.txt", "w", encoding="utf-8") as f:
    f.write("""RAG significa Retrieval-Augmented Generation. Es una técnica que
combina recuperación de información con generación de texto. Permite que los
LLMs accedan a información actualizada sin necesidad de reentrenamiento.""")

# PASO 2: Cargar los documentos
loader1 = TextLoader("doc1.txt", encoding="utf-8")
loader2 = TextLoader("doc2.txt", encoding="utf-8")
loader3 = TextLoader("doc3.txt", encoding="utf-8")
documents = loader1.load() + loader2.load() + loader3.load() # Cargo mis documentos creados

# PASO 3: Dividir documentos en chunks (chunkingco), en cadenas de texto, se hace por el limite de tokens
# Chunks más pequeños nos permiten búsquedas más precisas
text_splitter = CharacterTextSplitter(
    chunk_size=200, # 200 caracteres
    chunk_overlap=20 # El chunk overlap es la cantidad de texto (palabras o caracteres) que se repite
# entre dos chunks consecutivos para que no se pierda el contexto semántico cuando el texto se divide.
)
texts = text_splitter.split_documents(documents)
print(f"Documentos divididos en {len(texts)} chunks\n")

# PASO 4: Crear embeddings (incrustaciones, como hace un LLM) y almacenar en FAISS
# Básicamente es lo que hace cualquier LLM en su encoder, son representaciones vectorial densa, de N dimensiones
# Models embeddings es el modelo de Google que los crea
print("Creando vectorstore con embeddings de Google...")
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
vectorstore = FAISS.from_documents(texts, embeddings)  # Es como una base de datos de embeddings, si uso save_local(), lo guarda en disco
print("Vectorstore creado correctamente\n")

# PASO 5: Crear cadena de QA (Con esto construyo mi sistema RAG)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # stuff, combina todos los documentos y los combina en un bloque que se mete en el prompt del modelo
    retriever=vectorstore.as_retriever(),
    verbose=True
)

# PASO 6: Hacer preguntas ( esto son los CHAINS)
print("Pregunta 1: ¿Qué es LangChain?")
resultado1 = qa_chain.invoke({"query": "¿Qué es LangChain?"})
print(f"Respuesta: {resultado1['result']}\n")

print("Pregunta 2: ¿Qué pueden hacer los agentes?")
resultado2 = qa_chain.invoke({"query": "¿Qué pueden hacer los agentes?"})
print(f"Respuesta: {resultado2['result']}\n")

print("Pregunta 3: ¿Qué significa RAG?")
resultado3 = qa_chain.invoke({"query": "¿Qué significa RAG y para qué sirve?"})
print(f"Respuesta: {resultado3['result']}\n")




Creando documentos de ejemplo...
Documentos divididos en 3 chunks

Creando vectorstore con embeddings de Google...
Vectorstore creado correctamente

Pregunta 1: ¿Qué es LangChain?


> Entering new RetrievalQA chain...

> Finished chain.
Respuesta: LangChain es un framework para desarrollar aplicaciones con LLMs. Permite crear cadenas, agentes y sistemas RAG y es muy útil para construir aplicaciones inteligentes que necesitan acceso a datos externos.


Pregunta 2: ¿Qué pueden hacer los agentes?


> Entering new RetrievalQA chain...

> Finished chain.
Respuesta: Los agentes en LangChain pueden usar herramientas como búsqueda web, calculadoras y bases de datos. Un agente decide qué herramienta usar según la tarea que debe resolver.


Pregunta 3: ¿Qué significa RAG?


> Entering new RetrievalQA chain...

> Finished chain.
Respuesta: RAG significa Retrieval-Augmented Generation. Es una técnica que combina recuperación de información con generación de texto y permite que los LLMs accedan a 

 # EJERCICIO 4: AGENTE CON HERRAMIENTAS EXTERNAS
Un agente es un LLM que puede decidir qué herramientas usar
Proceso: Pregunta → Razonamiento → Elige herramienta → Ejecuta → Responde



In [12]:

# Imports necesarios para agentes
from langchain_community.tools.tavily_search import TavilySearchResults # Clase para buscar en la web
from langchain.agents import Tool, initialize_agent, AgentType # Tools para trabajar con las herramientas

# HERRAMIENTA 1: Búsqueda en Internet
search_tool = TavilySearchResults(
    max_results=3  # Número máximo de resultados a devolver
)

# HERRAMIENTA 2: Calculadora personalizada
# Define una función de Python que el Agente puede "llamar" para calcular.
def calculadora(expresion: str) -> str:
    """
    Evalúa expresiones matemáticas de forma segura.

    Args:
        expresion: Expresión matemática como string (ej: "15*23")

    Returns:
        Resultado del cálculo o mensaje de error
    """
    try:
        # eval() es peligroso, pero lo limitamos con __builtins__ vacío
        resultado = eval(expresion, {"__builtins__": {}}, {}) # evita que el código use funciones peligrosas del sistema
        return f"El resultado es: {resultado}"
    except Exception as e:
        return f"Error en el cálculo: {str(e)}"

# Definir las herramientas disponibles para el agente

tools = [
    Tool(
        name="Búsqueda Web",  # Nombre que verá el agente
        func=search_tool.run,  # Función a ejecutar, en este caso la búsqueda web
        description="Útil para buscar información actual en internet sobre noticias, eventos recientes, o datos que cambian frecuentemente"
    ),
    Tool(
        name="Calculadora", # Nombre que ve el agente
        func=calculadora,
        description="Útil para hacer cálculos matemáticos. Input debe ser una expresión matemática válida, por ejemplo: 15*23 o 100/4"
    )
]

# Crear el agente
print("\nCreando agente con herramientas...")
agent = initialize_agent(
    tools=tools,  # Herramientas disponibles
    llm=llm,  # Modelo de lenguaje Gemini
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,  # Tipo de agente React, piensa, ejecuta, observa
    verbose=True,  # Mostrar todo el razonamiento
    handle_parsing_errors=True  # Manejar errores automáticamente, si equivoca gerando el codiog
)
print("Agente creado correctamente\n")

# Probar el agente con un cálculo
print("Tarea 1: ¿Cuánto es 15 * 23?")
# Forma moderna de interactuar con el agente, aunque aqui funciona como un agente stateless, sin memoria para la conversación
resultado1 = agent.invoke({"input": "¿Cuánto es 15 multiplicado por 23?"})
print(f"Respuesta: {resultado1['output']}\n")

# Probar el agente con búsqueda web
print("Tarea 2: Buscar noticias sobre IA")
resultado2 = agent.invoke({"input": "¿Qué noticias importantes hay hoy sobre inteligencia artificial?"})
print(f"Respuesta: {resultado2['output']}\n")



Creando agente con herramientas...
Agente creado correctamente

Tarea 1: ¿Cuánto es 15 * 23?


> Entering new AgentExecutor chain...


/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.3.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(


I need to calculate the product of 15 and 23.
Action: Calculadora
Action Input: 15*23
Observation: El resultado es: 345
Thought:I now know the final answer.
Final Answer: 345


> Finished chain.
Respuesta: 345

Tarea 2: Buscar noticias sobre IA


> Entering new AgentExecutor chain...
I need to find out what the important news regarding artificial intelligence is today. I will use a web search to find the latest news.
Action: Búsqueda Web
Action Input: noticias inteligencia artificial hoy
Observation: [{'url': 'https://www.youtube.com/watch?v=o3ynxrSO3as', 'content': 'Bienvenidos a Anacrolbrum, el lugar donde la tecnología y la imaginación se encuentran. Hoy viajamos a octubre de 2025, un mes marcado por una avalancha de avances en inteligencia artificial, robótica e informática que están redefiniendo el modo en que trabajamos, pensamos y convivimos con las máquinas. En el terreno de la informática, Microsoft pone punto final al soporte de Windows 10, marcando el cierre de una era digit

# EJERCICIO 5: AGENTE COMPLETO (LLM + MEMORIA + RAG + HERRAMIENTAS)
Combinamos todo lo aprendido en un agente super completo
Capacidades: Recuerda conversación + Busca en docs + Usa internet + Calcula


In [13]:
# HERRAMIENTA PERSONALIZADA: RAG sobre documentos internos
def buscar_en_docs(pregunta: str) -> str:
    """
    Busca información en la base de conocimiento interna.

    Args:
        pregunta: Pregunta del usuario

    Returns:
        Respuesta basada en los documentos o mensaje de error
    """
    try:
        # Aquí se invoca la cadena RAG que habíamos hecho anteriormente (qa_chain) con la pregunta del usuario.
        # El Agente decide usar esta herramienta cuando la pregunta es sobre "conocimiento interno".
        resultado = qa_chain.invoke({"query": pregunta})
        return resultado.get("result", "No se encontró información relevante")
    except Exception as e:
        return f"Error al buscar en documentos: {str(e)}"

# Definir todas las herramientas juntas
tools_completo = [
    Tool(
        name="Base de Conocimiento",
        func=buscar_en_docs,
        description="Busca información en documentos internos sobre LangChain, agentes y RAG. Usar cuando se pregunte sobre conceptos técnicos de estos temas."
    ),
    Tool(
        name="Búsqueda Web",
        func=search_tool.run,
        description="Busca información actual en internet sobre noticias, eventos recientes"
    ),
    Tool(
        name="Calculadora",
        func=calculadora,
        description="Hace cálculos matemáticos. Input: expresión matemática"
    )
]

# Crear memoria conversacional para el agente
memory_agente = ConversationBufferMemory(
    memory_key="chat_history",  # Nombre de la variable en el prompt
    return_messages=True  # Devolver mensajes estructurados
)

# Crear agente completo con memoria
print("\nCreando agente completo con memoria...")
agente_completo = initialize_agent(
    tools=tools_completo,
    llm=llm,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,  # Agente conversacional
    memory=memory_agente,  # Memoria para recordar conversación
    verbose=True,
    handle_parsing_errors=True
)
print("Agente completo creado correctamente\n")

# Secuencia de interacciones que demuestra TODAS las capacidades

# 1. Memoria: Guardar información del usuario
print("Usuario: Hola, soy profesor de IA y doy clases sobre agentes")
resultado1 = agente_completo.invoke({
    "input": "Hola, soy profesor de IA y doy clases sobre agentes"
})
print(f"Asistente: {resultado1['output']}\n")

# 2. RAG: Buscar en documentos internos
print("Usuario: ¿Qué es LangChain según tus documentos internos?")
resultado2 = agente_completo.invoke({
    "input": "¿Qué es LangChain según tus documentos internos?"
})
print(f"Asistente: {resultado2['output']}\n")

# 3. Búsqueda Web: Información actual de internet
print("Usuario: ¿Y qué noticias hay hoy sobre este framework?")
resultado3 = agente_completo.invoke({
    "input": "¿Y qué noticias hay hoy sobre LangChain o herramientas similares?"
})
print(f"Asistente: {resultado3['output']}\n")

# 4. Calculadora: Resolver operaciones matemáticas
print("Usuario: Si tengo 30 estudiantes y quiero dividirlos en 5 grupos, ¿cuántos hay por grupo?")
resultado4 = agente_completo.invoke({
    "input": "Si tengo 30 estudiantes y quiero dividirlos en 5 grupos, ¿cuántos estudiantes hay por grupo?"
})
print(f"Asistente: {resultado4['output']}\n")

# 5. Memoria: Recordar información anterior
print("Usuario: ¿Recuerdas cuál es mi profesión?")
resultado5 = agente_completo.invoke({
    "input": "¿Recuerdas cuál es mi profesión?"
})
print(f"Asistente: {resultado5['output']}\n")




Creando agente completo con memoria...
Agente completo creado correctamente

Usuario: Hola, soy profesor de IA y doy clases sobre agentes


> Entering new AgentExecutor chain...
```json
{
    "action": "Final Answer",
    "action_input": "Hola, encantado de conocerte. Si tienes alguna pregunta sobre agentes de IA o necesitas ayuda con tus clases, no dudes en preguntar."
}
```

> Finished chain.
Asistente: Hola, encantado de conocerte. Si tienes alguna pregunta sobre agentes de IA o necesitas ayuda con tus clases, no dudes en preguntar.

Usuario: ¿Qué es LangChain según tus documentos internos?


> Entering new AgentExecutor chain...
```json
{
    "action": "Base de Conocimiento",
    "action_input": "¿Qué es LangChain?"
}
```

> Entering new RetrievalQA chain...

> Finished chain.

Observation: LangChain es un framework para desarrollar aplicaciones con LLMs. Permite crear cadenas, agentes y sistemas RAG. Es muy útil para construir aplicaciones inteligentes que necesitan acceso a dato